# Metodologia CRISP-DM - Proyecto Datos Empresas (RUES + SECOP + CIIU + TRM)

Documentacion de como las **6 fases de CRISP-DM** (*Cross-Industry Standard Process for Data Mining*) se aplicaron a este proyecto de principio a fin, con **ejemplos ejecutables** sobre las tablas reales del catalogo `datos_empresas`.

**Diferencia importante con un proyecto predictivo:** este es un proyecto de **analitica descriptiva**, no de machine learning. No hay una fase de entrenamiento de un modelo estadistico. La fase 4 (Modeling) se interpreta aqui como el diseno del **modelo analitico de datos** (dimension + hechos + agregados en Gold), que es el equivalente de "modelado" para un proyecto descriptivo. Se deja explicito en esa seccion para no simular un paso que no se hizo.

| Fase CRISP-DM | Entregable / Trabajo Practico | Que se hizo |
|---|---|---|
| 1. Business Understanding | Alcance del proyecto | Pregunta de negocio y criterio de exito |
| 2. Data Understanding | TP1 - Capa Bronze | Ingesta cruda de 4 fuentes abiertas |
| 3. Data Preparation | TP2 - Capas Silver y Gold | Reglas de calidad (DQL), MERGE idempotente, integracion |
| 4. Modeling | TP2 - Capa Gold | Diseno del modelo analitico (dimension, hechos, agregados) |
| 5. Evaluation | TP3 - Analitica descriptiva | Validaciones tecnicas automaticas + respuesta a la pregunta de negocio |
| 6. Deployment & Governance | TP3 - Dashboard, Job, Gobernanza | Streamlit, Job de Databricks, politicas de seguridad, auditoria |

---
## Fase 1 - Business Understanding (Entendimiento del Negocio)

**Concepto de la fase:** definir los objetivos de negocio, evaluar la situacion inicial y establecer las metas tecnicas.

**Aplicacion en el proyecto:**

* **Objetivo de negocio:** entender el tejido empresarial colombiano (RUES) y su relacion con la contratacion publica (SECOP), para identificar si existen factores macroeconomicos (TRM) o estructurales que esten asociados con la creacion y el cierre de empresas.
* **Pregunta de negocio central (la que responde este TP3):** ¿como se relaciona el comportamiento de las empresas colombianas -su creacion y cancelacion- con su participacion como proveedores del Estado, y con factores macroeconomicos como la TRM?
* **Criterio de exito (la "V de Valor"):** generar analitica descriptiva accionable sobre la capa Gold -no se promete un modelo predictivo, se promete evidencia clara, trazable y honesta, incluso si un hallazgo es "no hay relacion fuerte".

**Hito del proyecto:** esta pregunta se definio antes de construir el dashboard o el analisis final; todo lo que sigue en este notebook y en el dashboard de Streamlit responde a ella.

---
## Fase 2 - Data Understanding (Entendimiento de los Datos)

**Concepto de la fase:** recolectar los datos iniciales, explorar sus propiedades y verificar su calidad inicial.

**Aplicacion en el proyecto (Trabajo Practico 1 - Capa Bronze):**

| Fuente | Que es | Tabla Bronze |
|---|---|---|
| RUES | Registro mercantil de empresas | `bronze.de_semiestructurado_registromercantil_api` |
| SECOP II | Contratos publicos | `bronze.de_semiestructurado_secop_contratos_api` |
| CIIU | Catalogo de actividades economicas | `bronze.de_semiestructurado_actividadeseconomicas_api` |
| TRM | Tasa de cambio COP/USD | `bronze.de_semiestructurado_tasacambio_api` |

**Exploracion inicial y hallazgos de calidad** (los mismos que despues se convirtieron en reglas formales de rechazo en la Fase 3): empresas con la identificacion compuesta solo por ceros, codigos CIIU que no existen en el catalogo, fechas de matricula con centinelas imposibles (como el 1 de enero del año 1), y fechas de cancelacion anteriores a la matricula. La celda siguiente reproduce esa exploracion contra el Bronze real.

In [ ]:
%sql
-- Fase 2 - Exploracion inicial de las 4 fuentes Bronze
SELECT 'RUES' AS fuente, COUNT(*) AS filas, NULL AS detalle FROM datos_empresas.bronze.de_semiestructurado_registromercantil_api
UNION ALL
SELECT 'SECOP', COUNT(*), NULL FROM datos_empresas.bronze.de_semiestructurado_secop_contratos_api
UNION ALL
SELECT 'CIIU', COUNT(*), NULL FROM datos_empresas.bronze.de_semiestructurado_actividadeseconomicas_api
UNION ALL
SELECT 'TRM', COUNT(*), NULL FROM datos_empresas.bronze.de_semiestructurado_tasacambio_api;

Resultado real (consultado en vivo): RUES 3.022.492 filas, SECOP 289.157, CIIU 499, TRM 273. Las cuatro fuentes se ingieren de forma incremental (Bronze crece todos los dias via el Job programado), excepto CIIU que es un catalogo estatico.

---
## Fase 3 - Data Preparation (Preparacion de los Datos)

**Concepto de la fase:** seleccionar, limpiar, construir atributos derivados e integrar datos de multiples fuentes.

**Aplicacion en el proyecto (Trabajo Practico 2 - Capas Silver y Gold):**

1. **Limpieza y filtro de calidad (Silver + DQL):** cada fuente tiene entre 3 y 6 reglas de calidad (campos obligatorios, identificacion normalizada, fechas coherentes, integridad referencial contra el catalogo CIIU, unicidad por version mas reciente). Los registros rechazados van al esquema `dql`, nunca se pierden: conservan la fila original de Bronze, el motivo exacto y la fecha de rechazo.
2. **Construccion e integracion (Gold):** `gold.de_dim_empresas` une RUES con CIIU; `gold.de_fact_contratos` une SECOP con la TRM vigente (por rango de fechas) y con la dimension de empresas (para saber si el proveedor esta en RUES); `gold.de_agg_indicadores_mes` une las tres fuentes Gold por mes.
3. **Escritura idempotente:** todas las tablas se escriben con `MERGE` por llave de negocio y deteccion de cambios por hash -no con sobrescritura completa- porque Bronze se carga de forma incremental.

La celda siguiente verifica, con datos reales, que Bronze = Silver + DQL en cada fuente (ningun registro se pierde ni se cuenta dos veces).

In [ ]:
%sql
-- Fase 3 - Verificacion Bronze = Silver + DQL (conciliacion real, no una muestra)
SELECT 'RUES' AS fuente,
  (SELECT COUNT(*) FROM datos_empresas.bronze.de_semiestructurado_registromercantil_api) AS bronze,
  (SELECT COUNT(*) FROM datos_empresas.silver.de_registromercantil) AS silver,
  (SELECT COUNT(*) FROM datos_empresas.dql.de_registromercantil_dql) AS dql
UNION ALL
SELECT 'SECOP',
  (SELECT COUNT(*) FROM datos_empresas.bronze.de_semiestructurado_secop_contratos_api),
  (SELECT COUNT(*) FROM datos_empresas.silver.de_secop_contratos),
  (SELECT COUNT(*) FROM datos_empresas.dql.de_secop_contratos_dql);

Resultado real: RUES 3.022.492 = 2.651.079 (Silver) + 371.413 (DQL). SECOP 289.157 = 249.666 (Silver) + 39.491 (DQL). Balance exacto en ambas fuentes -ningun registro se pierde ni se duplica.

---
## Fase 4 - Modeling (Diseno del modelo analitico)

**Concepto de la fase (adaptado a un proyecto descriptivo):** en un proyecto de mineria de datos clasico esta fase entrena un modelo predictivo. Aqui, al ser un proyecto de **analitica descriptiva**, "modelar" significa disenar el **modelo de datos analitico** -un esquema en estrella simplificado- que estructura la informacion para que se pueda responder la pregunta de negocio con consultas directas, sin necesidad de un algoritmo de aprendizaje.

**Aplicacion en el proyecto (capa Gold):**

| Tabla | Tipo | Grano | Responde |
|---|---|---|---|
| `gold.de_dim_empresas` | Dimension | Una fila por empresa | Perfil, actividad economica, estado |
| `gold.de_fact_contratos` | Hecho | Una fila por contrato | Valor, modalidad, proveedor, enlace con RUES |
| `gold.de_agg_indicadores_mes` | Agregado | Una fila por mes | Serie de tiempo: empresas, contratacion, TRM |

Es un diseno deliberado: `fact_contratos` enlaza con `dim_empresas` para saber si un proveedor del Estado esta registrado en RUES, y `agg_indicadores_mes` une las tres fuentes por tiempo -la unica llave que comparten RUES/SECOP y TRM-, que es exactamente la estructura que necesita la pregunta de negocio de la Fase 1.

---
## Fase 5 - Evaluation (Evaluacion)

**Concepto de la fase:** evaluar si el resultado tecnico satisface la meta de negocio original antes de darlo por concluido.

**Evaluacion tecnica (automatica, notebook `09_Auditoria_Ejecucion`):** cada ejecucion del pipeline valida que no haya llaves duplicadas, que Bronze = Silver + DQL en cada fuente, y que Gold tenga exactamente las filas que le corresponden de Silver. Si algo no cuadra, la ejecucion falla. Esto ya se verifico en la Fase 3 con datos reales.

**Evaluacion de negocio (responde la pregunta de la Fase 1, con honestidad sobre el resultado):**

In [ ]:
%sql
-- Fase 5 - Evaluacion de negocio: TRM vs ciclo de vida de empresas, y cobertura proveedores-RUES
SELECT COUNT(*) AS meses_con_trm,
  ROUND(TRY_DIVIDE(COVAR_POP(trm_promedio, empresas_canceladas), STDDEV_POP(trm_promedio)*STDDEV_POP(empresas_canceladas)),3) AS corr_trm_cancelaciones,
  ROUND(TRY_DIVIDE(COVAR_POP(trm_promedio, empresas_matriculadas), STDDEV_POP(trm_promedio)*STDDEV_POP(empresas_matriculadas)),3) AS corr_trm_matriculas
FROM datos_empresas.gold.de_agg_indicadores_mes
WHERE trm_promedio IS NOT NULL;

In [ ]:
%sql
SELECT COUNT(*) AS contratos,
  SUM(CASE WHEN proveedor_registrado_rues THEN 1 ELSE 0 END) AS con_proveedor_rues,
  ROUND(100 * AVG(CASE WHEN proveedor_registrado_rues THEN 1 ELSE 0 END), 2) AS pct_con_proveedor_rues
FROM datos_empresas.gold.de_fact_contratos;

**Resultado real (15 meses con TRM disponible):** correlacion de Pearson TRM-promedio vs. empresas canceladas = **0,006**; TRM vs. empresas nuevas matriculadas = **0,02**. Ambas son practicamente cero.

**Conclusion de negocio, dicha sin maquillar el resultado:** con los datos y la ventana de tiempo disponibles (solo ~15 meses de TRM), **no hay evidencia de que la TRM influya en el cierre o la creacion de empresas**. No es un resultado negativo para el proyecto: es exactamente lo que se espera de una evaluacion honesta -la fase de Evaluation existe para decir "no" cuando los datos no sostienen la hipotesis, no para forzar una conclusion positiva. Si se quisiera profundizar, el siguiente paso del CRISP-DM seria volver a la Fase 2 (Data Understanding) a buscar mas historia de TRM, no inventar una relacion que no esta en los datos.

Sobre el otro componente de la pregunta de negocio: el **15,83%** de los contratos de SECOP tiene un proveedor identificado en el RUES cargado (39.521 de 249.666). Esto si es un hallazgo solido y explicable: la mayoria de proveedores del Estado son personas naturales con cedula (no tienen matricula mercantil), y el RUES que cargamos no cubre el 100% de las camaras de comercio del pais.

---
## Fase 6 - Deployment & Governance (Despliegue y Gobernanza)

**Concepto de la fase:** planificar el despliegue, establecer monitoreo, y documentar la gobernanza del dato en produccion.

**Aplicacion en el proyecto:**

* **Despliegue de resultados:** dashboard interactivo en **Streamlit** (`Trabajo Practico 3 - Analitica Descriptiva Gold/dashboard_streamlit/`), desplegable como Databricks App o standalone, que responde en vivo la pregunta de negocio de la Fase 1.
* **Orquestacion productiva:** Job de Databricks (`job_pipeline_datos_empresas.json`) con ejecucion diaria programada -la evidencia de idempotencia del TP2 ya mostro varias corridas reales sin duplicar datos.
* **Gobernanza operativa (Unity Catalog) — proteccion de datos bajo la Ley 1581 de 2012:**
  * **RBAC** con `GRANT`/`REVOKE` por grupo sobre el catalogo `datos_empresas`.
  * **Column masks** sobre columnas cuasi-identificadoras (`numero_identificacion_normalizado`, `nit_normalizado`).
  * **Row filters** por `camara_comercio`, para limitar el acceso de un analista a su region.
  * **Auditoria** de accesos via `system.access.audit`.
  * *Implementado en el notebook `02_Gobernanza_Seguridad.ipynb` de esta misma carpeta.*

---
## Matriz resumen para la sustentacion

| Fase CRISP-DM | Hito real del proyecto | Componente tecnico |
|---|---|---|
| 1. Business Understanding | Pregunta: TRM + proveedores del Estado + ciclo de vida empresarial | Documentacion de objetivos (este notebook) |
| 2. Data Understanding | Ingesta de RUES, SECOP, CIIU y TRM | Capa Bronze (TP1), 3.022.492 + 289.157 + 499 + 273 filas |
| 3. Data Preparation | Reglas de calidad DQL + MERGE idempotente + integracion | Capas Silver/DQL/Gold (TP2) |
| 4. Modeling | Diseno del modelo dimensional (dim + fact + agg) | Capa Gold (TP2) |
| 5. Evaluation | Validacion tecnica automatica + respuesta honesta a la pregunta de negocio | Auditoria (TP2) + este notebook (TP3) |
| 6. Deployment & Governance | Dashboard, Job programado, seguridad PII, auditoria de accesos | Streamlit + Job + `02_Gobernanza_Seguridad` (TP3) |

**Conclusion:** CRISP-DM es **iterativo**, no lineal. El hallazgo de la Fase 5 -correlacion casi nula entre TRM y el ciclo de vida de las empresas- es un ejemplo real de como un resultado de evaluacion puede regresar el proyecto a la Fase 1 para replantear la pregunta (por ejemplo, explorar otros factores macroeconomicos o esperar a tener mas historia de TRM), en vez de forzar una conclusion que los datos no respaldan.